# Feature Selection

## Imports

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mne
import warnings
warnings.filterwarnings('ignore')
from autoreject import AutoReject
from scipy.stats import kurtosis
import os
import requests
from collections import defaultdict
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, MinMaxScaler, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
from sklearn import svm
import pywt
import pickle
from sklearn.neighbors import KNeighborsClassifier
from sklearn.feature_selection import SelectKBest, RFE, f_classif, mutual_info_classif
from scipy.stats import entropy
from sklearn.decomposition import PCA

LOCAL_PATH = "/Users/khangnguyen/Documents/252_Final_Project/seizure_detection/data"
RF_MODEL = RandomForestClassifier(n_estimators = 100, random_state = 42)
SVM_MODEL = svm.SVC(kernel='rbf')
KNN_MODEL = KNeighborsClassifier(n_neighbors = 10)

In [2]:
time_features_df = pd.read_csv(f"{LOCAL_PATH}/csv/time_features.csv")
frequency_features_df = pd.read_csv(f"{LOCAL_PATH}/csv/frequency_features.csv")
dwt_features_df = pd.read_csv(f"{LOCAL_PATH}/csv/dwt_features.csv")
labels_df = pd.read_csv((f"{LOCAL_PATH}/csv/labels.csv"))

In [3]:
all_features = pd.merge(time_features_df, frequency_features_df, on = ['patient_id', 'file_id', 'epoch_index'], how = 'inner')
all_features = pd.merge(all_features, dwt_features_df, on = ['patient_id', 'file_id', 'epoch_index'], how = 'left')
all_features = pd.merge(all_features, labels_df, on = ['patient_id', 'file_id', 'epoch_index'], how = 'left')

In [4]:
id_cols = ['patient_id', 'file_id', 'epoch_index']

In [5]:
# splitting training and testing patient ids
all_patient_ids = all_features['patient_id'].unique()
train_ids, test_ids = train_test_split(all_patient_ids, test_size = 0.2, random_state = 42)

train_df = all_features[all_features['patient_id'].isin(train_ids)].reset_index(drop = True)
test_df = all_features[all_features['patient_id'].isin(test_ids)].reset_index(drop = True)

# getting X and y
x_train = train_df.drop(columns = id_cols + ['label'])
x_test = test_df.drop(columns = id_cols + ['label'])
y_train = train_df['label']
y_test = test_df['label']

In [6]:
def random_forest_predict(x_train, y_train, x_test, y_test):
    print("Model: Random Forest")
    label_encoder = LabelEncoder()
    y_train_encoded = label_encoder.fit_transform(y_train)
    y_test_encoded = label_encoder.transform(y_test)

    rf = RF_MODEL
    rf.fit(x_train, y_train_encoded)

    y_pred = rf.predict(x_test)
    print("Confusion Matrix:")
    print(confusion_matrix(y_test_encoded, y_pred))

    accuracy = accuracy_score(y_test_encoded, y_pred)
    print("Accuracy:", accuracy)

    precision = precision_score(y_test_encoded, y_pred)
    print("Precision:", precision)

    recall = recall_score(y_test_encoded, y_pred)
    print("Recall (Sensitivity):", recall)

    f1 = f1_score(y_test_encoded, y_pred)
    print("F1-Score:", f1)

In [7]:
def svm_predict(x_train, y_train, x_test, y_test):
    print("Model: SVM")
    label_encoder = LabelEncoder()
    y_train_encoded = label_encoder.fit_transform(y_train)
    y_test_encoded = label_encoder.transform(y_test)
    
    clf = SVM_MODEL
    clf.fit(x_train, y_train_encoded)
    y_pred_svm = clf.predict(x_test)

    print("Confusion Matrix:")
    print(confusion_matrix(y_test_encoded, y_pred_svm))

    accuracy = accuracy_score(y_test_encoded, y_pred_svm)
    print("Accuracy:", accuracy)

    precision = precision_score(y_test_encoded, y_pred_svm)
    print("Precision:", precision)

    recall = recall_score(y_test_encoded, y_pred_svm)
    print("Recall (Sensitivity):", recall)

    f1 = f1_score(y_test_encoded, y_pred_svm)
    print("F1-Score:", f1)

In [8]:
def knn_predict(x_train, y_train, x_test, y_test):
    print("Model: KNN")
    label_encoder = LabelEncoder()
    y_train_encoded = label_encoder.fit_transform(y_train)
    y_test_encoded = label_encoder.transform(y_test)

    knn = KNN_MODEL
    knn.fit(x_train, y_train_encoded)
    y_pred_knn = knn.predict(x_test)

    print("Confusion Matrix:")
    print(confusion_matrix(y_test_encoded, y_pred_knn))

    accuracy = accuracy_score(y_test_encoded, y_pred_knn)
    print("Accuracy:", accuracy)

    precision = precision_score(y_test_encoded, y_pred_knn)
    print("Precision:", precision)

    recall = recall_score(y_test_encoded, y_pred_knn)
    print("Recall (Sensitivity):", recall)

    f1 = f1_score(y_test_encoded, y_pred_knn)
    print("F1-Score:", f1)


## Variance Threshold

In [9]:
def filter_variance(x_train, x_test, threshold):
    scaler = MinMaxScaler()
    scaled_x_train_v1 = scaler.fit_transform(x_train)
    scaled_x_test_v1 = scaler.transform(x_test)

    scaled_x_train_v1 = pd.DataFrame(scaled_x_train_v1, columns=x_train.columns)
    scaled_x_test_v1 = pd.DataFrame(scaled_x_test_v1, columns=x_test.columns)
    
    variances = scaled_x_train_v1.var(axis=0)

    selected_columns = variances[variances >= threshold].index

    scaled_x_train_v1_filtered = scaled_x_train_v1[selected_columns]
    scaled_x_test_v1_filtered = scaled_x_test_v1[selected_columns]

    print(f"Original features: {x_train.shape[1]}")
    print(f"Selected features: {scaled_x_train_v1_filtered.shape[1]}")
    
    return scaled_x_train_v1_filtered, scaled_x_test_v1_filtered

## K-Best

In [12]:
def filter_kbest(x_train, y_train, x_test, k):
    selector = SelectKBest(score_func=f_classif, k=k)

    selector.fit(x_train, y_train)

    x_train_kbest = selector.transform(x_train)
    x_test_kbest = selector.transform(x_test)

    selected_feature_names = x_train.columns[selector.get_support()]
    
    x_train_kbest = pd.DataFrame(x_train_kbest, columns=selected_feature_names)
    x_test_kbest = pd.DataFrame(x_test_kbest, columns=selected_feature_names)

    print(f"Original features: {x_train.shape[1]}")
    print(f"Selected top {k} features: {x_train_kbest.shape[1]}")

    return x_train_kbest, x_test_kbest


## PCA

In [16]:
def filter_pca(x_train, x_test, n_components):
    scaler = StandardScaler()
    x_train_scaled = scaler.fit_transform(x_train)
    x_test_scaled = scaler.transform(x_test)

    pca = PCA(n_components=n_components)
    x_train_pca = pca.fit_transform(x_train_scaled)
    x_test_pca = pca.transform(x_test_scaled)

    component_names = [f"PC{i+1}" for i in range(n_components)]
    x_train_pca = pd.DataFrame(x_train_pca, columns=component_names)
    x_test_pca = pd.DataFrame(x_test_pca, columns=component_names)

    print(f"Original features: {x_train.shape[1]}")
    print(f"PCA reduced features: {x_train_pca.shape[1]}")
    print(f"Explained variance ratio (sum): {round(pca.explained_variance_ratio_.sum(), 4)}")

    return x_train_pca, x_test_pca

## Mutual Info

In [18]:
def filter_mutual_info(x_train, y_train, x_test, k):
    selector = SelectKBest(score_func=mutual_info_classif, k=k)

    selector.fit(x_train, y_train)

    x_train_mi = selector.transform(x_train)
    x_test_mi = selector.transform(x_test)

    selected_feature_names = x_train.columns[selector.get_support()]

    x_train_mi = pd.DataFrame(x_train_mi, columns=selected_feature_names)
    x_test_mi = pd.DataFrame(x_test_mi, columns=selected_feature_names)

    print(f"Original features: {x_train.shape[1]}")
    print(f"Selected top {k} features based on Mutual Information: {x_train_mi.shape[1]}")

    return x_train_mi, x_test_mi

In [19]:
x_train_variance_filtered, x_test_variance_filtered = filter_variance(x_train, x_test, 0.005)
x_train_kbest_filtered, x_test_kbest_filtered = filter_kbest(x_train_variance_filtered, y_train, x_test_variance_filtered, 50)
x_train_pca, x_test_pca = filter_pca(x_train_kbest_filtered, x_test_kbest_filtered, 20)
random_forest_predict(x_train_pca, y_train, x_test_pca, y_test)
print("-------------------------------------------")
svm_predict(x_train_pca, y_train, x_test_pca, y_test)
print("-------------------------------------------")
knn_predict(x_train_pca, y_train, x_test_pca, y_test)

Original features: 360
Selected features: 109
Original features: 109
Selected top 50 features: 50
Original features: 50
PCA reduced features: 20
Explained variance ratio (sum): 0.9735
Model: Random Forest
Confusion Matrix:
[[ 715 1772]
 [ 925 2854]]
Accuracy: 0.5695818704117459
Precision: 0.6169476869865975
Recall (Sensitivity): 0.7552262503307753
F1-Score: 0.6791195716835217
-------------------------------------------
Model: SVM
Confusion Matrix:
[[ 484 2003]
 [ 553 3226]]
Accuracy: 0.5920842642834344
Precision: 0.6169439663415567
Recall (Sensitivity): 0.8536649907382906
F1-Score: 0.7162522202486679
-------------------------------------------
Model: KNN
Confusion Matrix:
[[1237 1250]
 [1684 2095]]
Accuracy: 0.5317586977338015
Precision: 0.6263079222720478
Recall (Sensitivity): 0.5543794654670547
F1-Score: 0.5881527231892195


In [20]:
random_forest_predict(x_train, y_train, x_test, y_test)
print("-------------------------------------------")
svm_predict(x_train, y_train, x_test, y_test)
print("-------------------------------------------")
knn_predict(x_train, y_train, x_test, y_test)

Model: Random Forest
Confusion Matrix:
[[ 382 2105]
 [ 498 3281]]
Accuracy: 0.584583466326205
Precision: 0.6091719272187152
Recall (Sensitivity): 0.8682191055834877
F1-Score: 0.7159847244953628
-------------------------------------------
Model: SVM
Confusion Matrix:
[[  10 2477]
 [  25 3754]]
Accuracy: 0.6007022023619534
Precision: 0.6024715134007382
Recall (Sensitivity): 0.9933844932521831
F1-Score: 0.75004995004995
-------------------------------------------
Model: KNN
Confusion Matrix:
[[1190 1297]
 [1678 2101]]
Accuracy: 0.525215448451963
Precision: 0.6183048852266039
Recall (Sensitivity): 0.5559671870865308
F1-Score: 0.585481398913195
